## 数据整合 汇总全部原始序列，合并为“文件1”。

In [1]:
import os

# 指定文件夹路径
folder_path = r"C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences_1109"
output_file = os.path.join(folder_path, "merged_all_1109.fasta")

merged_files = []

with open(output_file, "w", encoding="utf-8") as outfile:
    for filename in sorted(os.listdir(folder_path)):
        if filename.lower().endswith(".fasta") and filename != os.path.basename(output_file):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as infile:
                contents = infile.read().strip()
                if contents:  # 跳过空文件
                    outfile.write(contents + "\n")
                    merged_files.append(filename)

# 输出简洁日志
print("✅ FASTA 合并完成")
print(f"输出文件：{output_file}")
print(f"共合并 {len(merged_files)} 个文件：")
for f in merged_files:
    print("  -", f)


✅ FASTA 合并完成
输出文件：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences_1109\merged_all_1109.fasta
共合并 3 个文件：
  - UNIREF100_merged.fasta
  - UNIREF50_merged.fasta
  - UNIREF90_merged.fasta


## 初步去冗余    以氨基酸序列完全一致为判据进行去重：对每组相同序列仅保留首个出现的条目，得到非冗余集合“文件1+”。

In [20]:
from collections import OrderedDict
from Bio import SeqIO
import os

# 输入输出路径
folder_path = r"C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences"
input_file = os.path.join(folder_path, "merged_all.fasta")
output_file = os.path.join(folder_path, "merged_all_plus.fasta")

# 使用 OrderedDict 保证保留第一个出现的序列
unique_records = OrderedDict()

# 逐条读取 fasta
for record in SeqIO.parse(input_file, "fasta"):
    seq_str = str(record.seq)
    if seq_str not in unique_records:
        unique_records[seq_str] = record  # 保留首个序列

# 输出去重后的结果
SeqIO.write(unique_records.values(), output_file, "fasta")

print("✅ 去冗余完成！")
print(f"输入文件：{input_file}")
print(f"输出文件：{output_file}")
print(f"原始序列数：{len(list(SeqIO.parse(input_file, 'fasta')))}")
print(f"去重后序列数：{len(unique_records)}")


✅ 去冗余完成！
输入文件：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\merged_all.fasta
输出文件：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\merged_all_plus.fasta
原始序列数：231926
去重后序列数：192514


## 筛选字符代码

In [9]:
# -*- coding: utf-8 -*-
"""
修复并合并 *_DomainHits_rep.txt：
1) 修复换行/断行，确保每条记录有 11 列（Tab 分隔）
2) 生成两个 CSV：
   - merged_full.csv（修复后全量）
   - merged_filtered.csv（删 Incomplete=N/C，且 Short name 命中 GMC_oxred_C/GMC_oxred_N/BBE/FAD）
3) 另存一份筛选后的 TXT：merged_results.txt（Tab 分隔）

使用方法：
    直接运行本脚本。按需修改 FOLDER 路径。
"""

import os
import io
import sys
import traceback
import pandas as pd

# ===== 配置区（按需修改） =====
FOLDER = r"C:\Users\localadmin\Desktop\其他重要\results_playwright"  # 你的文件夹
TXT_GLOB_SUFFIX = ".txt"  # 只处理 .txt
EXPECTED_COLS = 11        # 期望列数（DomainHits固定为11列）
TARGETS_REGEX = r"(?:GMC_oxred_C|GMC_oxred_N|BBE|FAD)"  # 关键词（不区分大小写）
FULL_CSV = "merged_full.csv"
FILTERED_CSV = "merged_filtered.csv"
FILTERED_TXT = "merged_results.txt"

# ===== 行修复读取器：把同一条记录被换行/截断的多行拼回去 =====
def load_repaired(path):
    """
    读取一个 *_DomainHits_rep.txt，忽略注释行(#)，把被换行/截断的记录拼回到 11 列（10个Tab）
    返回 DataFrame（列：Query, Hit type, PSSM-ID, From, To, E-Value, Bitscore, Accession, Short name, Incomplete, Superfamily）
    """
    rows = []
    header = None
    buf = ""

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            # 忽略注释/空行
            if raw.startswith("#") or raw.strip() == "":
                continue

            line = raw.rstrip("\n")

            # 第一行表头
            if header is None and line.startswith("Query"):
                header = line
                continue

            # 记录通常以 "Q#" 开头；遇到新记录时，先冲洗上一个缓冲
            if line.startswith("Q#") and buf:
                rows.append(buf)
                buf = line
            else:
                buf = (buf + line) if buf else line

            # 凑够列（11列=10个tab）就冲洗
            if buf.count("\t") >= EXPECTED_COLS - 1:
                rows.append(buf)
                buf = ""

        # 文件末尾可能还有残余
        if buf:
            rows.append(buf)

    # 兜底表头（极少数文件缺头行时）
    if header is None:
        header = "Query\tHit type\tPSSM-ID\tFrom\tTo\tE-Value\tBitscore\tAccession\tShort name\tIncomplete\tSuperfamily"

    data = "\n".join([header] + rows)
    df = pd.read_csv(io.StringIO(data), sep="\t", dtype=str, keep_default_na=False, engine="python")

    # 清理列名空格
    df.columns = [c.strip() for c in df.columns]
    return df


def main():
    if not os.path.isdir(FOLDER):
        print(f"❌ 找不到目录：{FOLDER}")
        sys.exit(1)

    all_txt = [fn for fn in os.listdir(FOLDER) if fn.lower().endswith(TXT_GLOB_SUFFIX)]
    if not all_txt:
        print(f"❌ 目录下未找到 {TXT_GLOB_SUFFIX} 文件：{FOLDER}")
        sys.exit(1)

    repaired_dfs = []
    ok_files = 0
    for fn in all_txt:
        path = os.path.join(FOLDER, fn)
        try:
            df = load_repaired(path)

            # 校验必要列
            need = {"Short name", "Incomplete"}
            if not need.issubset(df.columns):
                print(f"⚠️ 跳过（列缺失）：{fn} -> 现有列：{list(df.columns)}")
                continue

            repaired_dfs.append(df)
            ok_files += 1
        except Exception as e:
            print(f"❌ 读取失败：{fn} -> {e}")
            traceback.print_exc(limit=1)

    if not repaired_dfs:
        print("❗没有任何文件成功解析为表格，请检查源文件格式是否为 Tab 分隔且包含表头。")
        sys.exit(1)

    # # 合并全量（不筛选）
    # merged_full = pd.concat(repaired_dfs, ignore_index=True)
    # full_csv_path = os.path.join(FOLDER, FULL_CSV)
    # merged_full.to_csv(full_csv_path, index=False)
    # print(f"✅ 全量CSV已生成：{full_csv_path}  （共 {len(merged_full)} 行；合并 {ok_files} 个文件）")

    # 生成筛选版：删 Incomplete=N/C；Short name 命中关键词
    filtered = merged_full.copy()
    filtered = filtered[~filtered["Incomplete"].astype(str).str.strip().isin(["N", "C"])]
    filtered = filtered[filtered["Short name"].astype(str).str.contains(TARGETS_REGEX, regex=True, case=False, na=False)]

    # 保存 CSV + TXT（Tab分隔）
    filtered_csv_path = os.path.join(FOLDER, FILTERED_CSV)
    filtered.to_csv(filtered_csv_path, index=False)
    print(f"✅ 筛选CSV已生成：{filtered_csv_path}  （{len(filtered)} 行）")

    filtered_txt_path = os.path.join(FOLDER, FILTERED_TXT)
    filtered.to_csv(filtered_txt_path, sep="\t", index=False)
    print(f"✅ 筛选TXT已生成：{filtered_txt_path}  （{len(filtered)} 行）")

    if len(filtered) == 0:
        print("⚠️ 提示：筛选结果为空。可能原因：\n"
              "   1) Incomplete 大多是 N/C，被删光；\n"
              "   2) Short name 中没有 GMC_oxred_C/GMC_oxred_N/BBE/FAD；\n"
              "   3) 源文件不是标准 Tab 分隔或记录被严重截断，可把样例发我进一步定制修复规则。")

if __name__ == "__main__":
    main()


✅ 全量CSV已生成：C:\Users\localadmin\Desktop\其他重要\results_playwright\merged_full.csv  （共 285626 行；合并 215 个文件）
✅ 筛选CSV已生成：C:\Users\localadmin\Desktop\其他重要\results_playwright\merged_filtered.csv  （82072 行）
✅ 筛选TXT已生成：C:\Users\localadmin\Desktop\其他重要\results_playwright\merged_results.txt  （82072 行）


## ncbi 提取序列代码

In [6]:
# -*- coding: utf-8 -*-
import os, re, time, zipfile, shutil, asyncio
from datetime import datetime
from pathlib import Path
from playwright.async_api import async_playwright

# ========= 配置 =========
BASE_DIR = r"C:\Users\localadmin\Desktop\其他重要\1202"

ZIP_FILE = os.path.join(BASE_DIR, "output_sequences.zip")
INPUT_DIR = os.path.join(BASE_DIR, "input_fasta")
RESULT_DIR = os.path.join(BASE_DIR, "results_playwright")
FAILED_DIR = os.path.join(BASE_DIR, "failed_fasta_playwright")
LOG_DIR = os.path.join(BASE_DIR, "logs_playwright")

BASE_URL = "https://www.ncbi.nlm.nih.gov/Structure/bwrpsb/bwrpsb.cgi"
MAX_RETRY = 3
ANALYSIS_TIMEOUT = 60 * 30
HEADLESS = True

# ========= 新增：模式选择 =========
# 可选值： "rep" → Concise,  "std" → Standard,  "full" → Full
DATA_MODE = "rep"

# ========= 初始化 =========
for d in [INPUT_DIR, RESULT_DIR, FAILED_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

LOG_FILE = os.path.join(LOG_DIR, "run_summary.txt")
with open(LOG_FILE, "w", encoding="utf-8") as wf:
    wf.write(f"Batch CD-search run started: {datetime.now()}\n")

# ========= 解压 ZIP =========
print("正在解压 ZIP 文件...")
if os.path.exists(ZIP_FILE):
    with zipfile.ZipFile(ZIP_FILE, "r") as zf:
        zf.extractall(INPUT_DIR)
        print(f"✅ 已解压到 {INPUT_DIR}")
else:
    print(f"⚠️ 未找到 ZIP：{ZIP_FILE}（将直接扫描目录）")

fasta_files = [f for f in os.listdir(INPUT_DIR)
               if f.lower().endswith(".fasta") or f.lower().endswith(".fasta.txt")]
if not fasta_files:
    print("❌ 未找到任何 FASTA")
    raise SystemExit
print(f"✅ 找到 {len(fasta_files)} 个 FASTA")

# ========= 日志函数 =========
def log(global_msg: str, file_log: str = None):
    print(global_msg)
    with open(LOG_FILE, "a", encoding="utf-8") as wf:
        wf.write(f"{datetime.now().isoformat()}  {global_msg}\n")
    if file_log:
        with open(file_log, "a", encoding="utf-8") as wf:
            wf.write(f"{datetime.now().isoformat()}  {global_msg}\n")

# ========= 提交逻辑 =========
async def robust_submit(page, file_log):
    form_sel = "form[action*='bwrpsb.cgi'], form[name='bwrpsb'], form"
    try:
        await page.eval_on_selector(form_sel, "f => f && f.removeAttribute('target')")
    except Exception:
        pass
    try:
        async with page.expect_navigation(wait_until="load", timeout=60000):
            await page.evaluate("""
            (sel)=>{
                const f = document.querySelector(sel) || document.querySelector('form');
                if(!f) throw new Error('no form');
                if (typeof window.DoSubmit === 'function') window.DoSubmit();
                else f.submit();
            }""", form_sel)
        log("✅ 已提交并导航", file_log)
        return True
    except Exception as e:
        log(f"⚠️ 提交后无导航：{e}", file_log)
    for sel in [
        "input[type='submit']","button[type='submit']",
        "input[value*='Submit' i]","input[value*='Search' i]","input[value*='CD-Search' i]",
        "button:has-text('Submit')","button:has-text('Search')","button:has-text('CD-Search')",
        "input[name='search']",
    ]:
        try:
            loc = page.locator(sel).first
            if await loc.count():
                async with page.expect_navigation(wait_until="load", timeout=60000):
                    await loc.click(force=True, timeout=5000)
                log(f"🖱️ 已点击提交按钮: {sel}", file_log)
                return True
        except Exception:
            continue
    return False

# ========= 等待完成 =========
async def wait_for_search_completed(page, file_log, timeout_sec=1800):
    log("⌛ 等待出现 'Search completed successfully' ...", file_log)
    t0 = time.time()
    while time.time() - t0 < timeout_sec:
        if await page.locator("text=Search completed successfully").first.count():
            log("✅ 看到完成提示：Search completed successfully", file_log)
            try:
                await page.wait_for_selector("text=Select Download data", timeout=30_000)
                log("✅ 看到下载选项区域：Select Download data", file_log)
            except Exception:
                log("⚠️ 没直接看到 'Select Download data'", file_log)
            return True
        await asyncio.sleep(2.0)
    log("❌ 等待完成提示超时", file_log)
    return False

# ========= 激活下载区（支持模式切换） =========
async def prep_download_area(page, file_log):
    log(f"🧩 选择 Domain Hits + Data mode={DATA_MODE.upper()}，并启用 Download 按钮", file_log)
    js = rf"""
    () => {{
      const hits = document.querySelector('#hid_tdata_hits');
      if (hits) {{ hits.disabled = false; hits.checked = true; hits.dispatchEvent(new Event('change',{{bubbles:true}})); }}

      const modeMap = {{
        rep: document.querySelector('#hid_dmode_rep'),
        std: document.querySelector('#hid_dmode_std'),
        full: document.querySelector('#hid_dmode_full')
      }};
      const wanted = '{DATA_MODE.lower()}';
      const mode = modeMap[wanted] || modeMap.full;
      if (mode) {{
        mode.disabled = false;
        mode.checked = true;
        mode.dispatchEvent(new Event('click',{{bubbles:true}}));
        mode.dispatchEvent(new Event('change',{{bubbles:true}}));
      }}

      const btn = document.querySelector('input[type=image][src*="download"]');
      if (btn) {{ btn.disabled = false; btn.style.pointerEvents='auto'; btn.style.opacity='1'; btn.id='__html_download_btn__'; }}

      return {{ ok: !!(hits && hits.checked && btn), mode: wanted }};
    }}
    """
    try:
        st = await page.evaluate(js)
        log(f"✅ 激活完成 DomainHits 模式={st['mode']} ok={st['ok']}", file_log)
    except Exception as e:
        log(f"⚠️ prep_download_area 执行失败: {e}", file_log)

# ========= 下载逻辑 =========
async def download_after_wait(context, page, out_path, file_log):
    log(f"⬇️ 尝试触发网页端 Domain Hits 下载（模式={DATA_MODE.upper()}）...", file_log)
    btn = await page.query_selector("input[type=image][src*='download']")
    if not btn:
        log("❌ 页面中未找到 Download 按钮", file_log)
        return False

    try:
        async with page.expect_download(timeout=180_000) as dl_info:
            await btn.click(force=True, delay=200)
        download = await dl_info.value
        await download.save_as(out_path)
        log(f"💾 成功保存真实下载文件：{out_path}", file_log)
        return True
    except Exception as e:
        log(f"⚠️ 未检测到浏览器下载事件 ({e})，尝试后台下载接口...", file_log)

    try:
        cdsid = await page.evaluate("""
        () => {
            const el = document.querySelector('#id_cdsid') ||
                       document.querySelector('input[name="cdsid"]') ||
                       document.querySelector('#hid_DlRid');
            return el ? (el.value || el.textContent || '').trim() : '';
        }
        """)
        if not cdsid:
            html = await page.content()
            m = re.search(r"cdsid[=:\s'\"]([\w\-]+)", html)
            cdsid = m.group(1) if m else ""
        if not cdsid:
            log("❌ 无法提取 cdsid", file_log)
            return False

        log(f"✅ 捕获到 cdsid: {cdsid}", file_log)
        url = f"https://www.ncbi.nlm.nih.gov/Structure/bwrpsb/bwrpsb.cgi?cdsid={cdsid}&tdata=hits&dmode={DATA_MODE}"

        for i in range(12):
            response = await context.request.get(url)
            text = await response.text()
            if "#status\t0" in text or "hits" in text:
                Path(out_path).write_text(text, encoding="utf-8")
                log(f"💾 成功保存 Domain Hits 结果：{out_path}", file_log)
                return True
            log(f"⌛ 文件尚未生成（尝试 {i+1}/12）...等待 10s 再试", file_log)
            await asyncio.sleep(10)

        log("❌ 超时未生成结果", file_log)
        return False
    except Exception as e:
        log(f"❌ 后台下载失败: {e}", file_log)
        return False

# ========= 单文件执行 =========
async def process_one_file(play, fasta_file: str):
    fasta_path = os.path.join(INPUT_DIR, fasta_file)
    stem = Path(fasta_file).stem
    file_log = os.path.join(LOG_DIR, f"{stem}.log")

    # ========= 新增：跳过已存在逻辑 =========
    # 构造预期的输出文件名，需与后续下载保存时的命名规则完全一致
    expected_out_path = os.path.join(RESULT_DIR, f"{stem}_DomainHits_{DATA_MODE}.txt")
    
    if os.path.exists(expected_out_path):
        log(f"⏭️ 结果文件已存在，跳过处理: {fasta_file}", file_log)
        return True
    # =====================================

    for attempt in range(1, MAX_RETRY + 1):
        log(f"\n[{fasta_file}] 尝试 {attempt}/{MAX_RETRY}", file_log)
        browser = context = page = None
        try:
            browser = await play.chromium.launch(headless=HEADLESS)
            context = await browser.new_context(accept_downloads=True)
            page = await context.new_page()
            await page.goto(BASE_URL, timeout=60000)
            log("🌍 页面已打开", file_log)

            await page.set_input_files("input[type=file]", fasta_path)
            log(f"📤 已选择文件：{fasta_file}", file_log)

            if not await robust_submit(page, file_log):
                raise RuntimeError("❌ 提交失败")

            if not await wait_for_search_completed(page, file_log, ANALYSIS_TIMEOUT):
                raise RuntimeError("分析未在超时内完成")

            await prep_download_area(page, file_log)
            await asyncio.sleep(1)

            # 此处路径变量即为上方检查的 expected_out_path
            out_path = expected_out_path 
            ok = await download_after_wait(context, page, out_path, file_log)
            if not ok:
                raise RuntimeError("未能成功保存结果")

            await context.close()
            await browser.close()
            log("✅ 文件处理完成", file_log)
            return True

        except Exception as e:
            log(f"❗ 错误: {e}", file_log)
            try:
                if page:
                    ss = os.path.join(LOG_DIR, f"{stem}_err_{attempt}.png")
                    await page.screenshot(path=ss, full_page=True)
                    log(f"🖼️ 错误截图: {ss}", file_log)
            except Exception:
                pass
            await asyncio.sleep(3)
            continue

    log(f"❌ {fasta_file} 多次失败，移动到失败目录", file_log)
    try:
        shutil.copy2(fasta_path, FAILED_DIR)
    except Exception as e:
        log(f"移动失败: {e}", file_log)
    return False

# ========= 主入口 =========
async def main(debug_first_only=True):
    async with async_playwright() as p:
        if debug_first_only:
            await process_one_file(p, fasta_files[0])
        else:
            for f in fasta_files:
                await process_one_file(p, f)

await main(debug_first_only=False)


正在解压 ZIP 文件...
✅ 已解压到 C:\Users\localadmin\Desktop\其他重要\1202\input_fasta
✅ 找到 321 个 FASTA
⏭️ 结果文件已存在，跳过处理: split_1.fasta
⏭️ 结果文件已存在，跳过处理: split_10.fasta
⏭️ 结果文件已存在，跳过处理: split_100.fasta
⏭️ 结果文件已存在，跳过处理: split_101.fasta
⏭️ 结果文件已存在，跳过处理: split_102.fasta
⏭️ 结果文件已存在，跳过处理: split_103.fasta
⏭️ 结果文件已存在，跳过处理: split_104.fasta
⏭️ 结果文件已存在，跳过处理: split_105.fasta
⏭️ 结果文件已存在，跳过处理: split_106.fasta
⏭️ 结果文件已存在，跳过处理: split_107.fasta
⏭️ 结果文件已存在，跳过处理: split_108.fasta
⏭️ 结果文件已存在，跳过处理: split_109.fasta
⏭️ 结果文件已存在，跳过处理: split_11.fasta
⏭️ 结果文件已存在，跳过处理: split_110.fasta
⏭️ 结果文件已存在，跳过处理: split_111.fasta
⏭️ 结果文件已存在，跳过处理: split_112.fasta
⏭️ 结果文件已存在，跳过处理: split_113.fasta
⏭️ 结果文件已存在，跳过处理: split_114.fasta
⏭️ 结果文件已存在，跳过处理: split_115.fasta
⏭️ 结果文件已存在，跳过处理: split_116.fasta
⏭️ 结果文件已存在，跳过处理: split_117.fasta
⏭️ 结果文件已存在，跳过处理: split_118.fasta
⏭️ 结果文件已存在，跳过处理: split_119.fasta
⏭️ 结果文件已存在，跳过处理: split_12.fasta
⏭️ 结果文件已存在，跳过处理: split_120.fasta
⏭️ 结果文件已存在，跳过处理: split_121.fasta
⏭️ 结果文件已存在，跳过处理: split_122.fasta
⏭️ 结果文件已存在，跳过处理: split_12

## 下载获得序列代码

In [1]:
import os
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

# === 参数设置 ===
query = "dihydroxy acid dehydratase"
save_dir = r"C:\Users\localadmin\Desktop\其他重要\11.24\EBI\dihydroxy acid dehydratase"
os.makedirs(save_dir, exist_ok=True)

# 可切换数据库
databases = ["UniRef100", "UniRef90", "UniRef50", "UniProtKB", "EPO", "JPO", "USPTO"]

# === 获取 UniRef ID 列表（增加缓存机制） ===
def get_uniref_ids(db: str, query: str, page_size: int = 5000):
    cache_path = os.path.join(save_dir, f"{db}_ids.txt")

    # 若缓存存在，直接读取并返回
    if os.path.exists(cache_path):
        print(f"📁 发现缓存：{cache_path}，跳过 ID 重新下载")
        with open(cache_path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]

    print(f"🔍 获取 {db} ID 列表中 ...")
    ids = []
    start = 0

    while True:
        params = {"query": query, "format": "idlist", "size": page_size, "start": start}
        url = f"https://www.ebi.ac.uk/ebisearch/ws/rest/{db}"
        r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code != 200:
            print(f"❌ 获取 {db} ID 失败：HTTP {r.status_code}")
            break

        new_ids = [line.strip() for line in r.text.splitlines() if line.strip()]
        if not new_ids:
            break

        ids.extend(new_ids)
        print(f"📥 已获取 {len(new_ids)} 条 (总计 {len(ids)})")

        if len(new_ids) < page_size:
            break

        start += page_size
        time.sleep(0.5)

    # 写入缓存
    with open(cache_path, "w", encoding="utf-8") as f:
        f.write("\n".join(ids))

    return ids


# === 下载单条 FASTA ===
def fetch_fasta(uid, db_type):
    if db_type.lower() == "uniprotkb":
        url = f"https://rest.uniprot.org/uniprotkb/{uid}.fasta"
    else:
        url = f"https://rest.uniprot.org/uniref/{uid}.fasta"
    
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        if r.status_code == 200 and r.text.startswith(">"):
            return r.text
    except:
        pass

    return None


# === 并行下载（加强续跑） ===
def download_uniref_parallel(db, ids, max_workers=10, progress_interval=50):
    db_dir = os.path.join(save_dir, db)
    os.makedirs(db_dir, exist_ok=True)

    # 检查已存在的 fasta
    existing = set(f[:-6] for f in os.listdir(db_dir) if f.endswith(".fasta"))

    # 检查是否存在失败记录
    failed_path = os.path.join(save_dir, f"{db}_failed_ids.txt")
    if os.path.exists(failed_path):
        with open(failed_path, "r", encoding="utf-8") as f:
            previous_failed = [line.strip() for line in f if line.strip()]
        print(f"🔁 发现之前下载失败的 {len(previous_failed)} 条，将优先重试")
        ids = previous_failed + [uid for uid in ids if uid not in existing and uid not in previous_failed]
    else:
        ids = [uid for uid in ids if uid not in existing]

    print(f"🔁 续跑模式：剩余 {len(ids)} 条未下载。")

    fail_ids = []
    success = 0
    total = len(ids)

    print(f"\n⬇️ 开始下载 {db.upper()} 的 FASTA 序列 ({total} 条 ID) ...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_fasta, uid, db): uid for uid in ids}
        
        for i, future in enumerate(as_completed(futures), 1):
            uid = futures[future]
            fasta = future.result()

            if fasta:
                with open(os.path.join(db_dir, f"{uid}.fasta"), "w", encoding="utf-8") as f:
                    f.write(fasta)
                success += 1
            else:
                fail_ids.append(uid)

            if i % progress_interval == 0 or i == total:
                print(f"📦 进度 {i}/{total} | 成功 {success} | 失败 {len(fail_ids)}")

    # 更新失败记录
    with open(failed_path, "w", encoding="utf-8") as f:
        f.write("\n".join(fail_ids))

    # 合并所有 fasta，避免重复
    merged_path = os.path.join(save_dir, f"{db.upper()}_merged.fasta")
    seq_set = set()
    with open(merged_path, "w", encoding="utf-8") as merged:
        for fname in sorted(os.listdir(db_dir)):
            if fname.endswith(".fasta"):
                with open(os.path.join(db_dir, fname), "r", encoding="utf-8") as f:
                    content = f.read().strip()
                    if content not in seq_set:
                        seq_set.add(content)
                        merged.write(content + "\n\n")

    print(f"✅ {db.upper()} 下载完成: 成功 {success} / 失败 {len(fail_ids)}")
    print(f"💾 合并文件保存于：{merged_path}\n")


# === 主程序 ===
for db in databases:
    ids = get_uniref_ids(db, query)
    if not ids:
        print(f"⚠️ {db} 无结果。")
        continue
    download_uniref_parallel(db, ids)

print("\n🎉 所有数据库下载完成！")


🔍 获取 UniProtKB ID 列表中 ...



KeyboardInterrupt



In [2]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

# === 参数设置 ===
save_dir = r"C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences"
os.makedirs(save_dir, exist_ok=True)

# 要重试的失败文件
fail_files = [
    "uniref50_failed_ids.txt",
    "uniref90_failed_ids.txt",
    "uniref100_failed_ids.txt",
]

# === 下载函数 ===
def fetch_fasta(uid):
    url = f"https://rest.uniprot.org/uniref/{uid}.fasta"
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        if r.status_code == 200 and r.text.startswith(">"):
            return r.text
    except Exception:
        pass
    return None


# === 重试下载函数 ===
def retry_failed_ids(fail_path, max_workers=10, progress_interval=20):
    db = os.path.basename(fail_path).split("_")[0]  # uniref50 -> db
    db_upper = db.upper()

    db_dir = os.path.join(save_dir, db)
    merged_path = os.path.join(save_dir, f"{db_upper}_merged.fasta")

    os.makedirs(db_dir, exist_ok=True)

    with open(fail_path, "r", encoding="utf-8") as f:
        ids = [line.strip() for line in f if line.strip()]
    total = len(ids)

    print(f"\n🔁 开始重新下载 {db_upper} 失败序列 ({total} 条)...")

    success = 0
    new_fail = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_fasta, uid): uid for uid in ids}
        for i, future in enumerate(as_completed(futures), 1):
            uid = futures[future]
            fasta = future.result()
            if fasta:
                # 保存单条
                file_path = os.path.join(db_dir, f"{uid}.fasta")
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(fasta)
                # ✅ 追加写入总合并文件
                with open(merged_path, "a", encoding="utf-8") as merged:
                    merged.write(fasta + "\n")
                success += 1
            else:
                new_fail.append(uid)

            if i % progress_interval == 0 or i == total:
                print(f"📦 进度 {i}/{total} | 成功 {success} | 失败 {len(new_fail)}")

    # 更新失败列表
    if new_fail:
        with open(fail_path, "w", encoding="utf-8") as f:
            f.write("\n".join(new_fail))
        print(f"⚠️ 仍有 {len(new_fail)} 条下载失败，已更新到 {fail_path}")
    else:
        os.remove(fail_path)
        print(f"✅ {db_upper} 所有失败条目已成功补全，删除失败列表文件。")

    print(f"💾 已追加更新到：{merged_path}\n")


# === 主程序 ===
for file in fail_files:
    fail_path = os.path.join(save_dir, file)
    if os.path.exists(fail_path):
        retry_failed_ids(fail_path, max_workers=10)
    else:
        print(f"⚠️ 未找到 {file}，跳过。")

print("🎉 所有失败序列已重试并追加至对应合并文件！")



🔁 开始重新下载 UNIREF50 失败序列 (3 条)...
📦 进度 3/3 | 成功 3 | 失败 0
✅ UNIREF50 所有失败条目已成功补全，删除失败列表文件。
💾 已追加更新到：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\UNIREF50_merged.fasta


🔁 开始重新下载 UNIREF90 失败序列 (9 条)...
📦 进度 9/9 | 成功 9 | 失败 0
✅ UNIREF90 所有失败条目已成功补全，删除失败列表文件。
💾 已追加更新到：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\UNIREF90_merged.fasta


🔁 开始重新下载 UNIREF100 失败序列 (8 条)...
📦 进度 8/8 | 成功 8 | 失败 0
✅ UNIREF100 所有失败条目已成功补全，删除失败列表文件。
💾 已追加更新到：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\UNIREF100_merged.fasta

🎉 所有失败序列已重试并追加至对应合并文件！


## JGI下载获得序列代码

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import re

# 输入 ID 文件
input_file = r"C:\Users\localadmin\Desktop\其他重要\JGI\id.txt"

# 输出文件路径
folder_path = r"C:\Users\localadmin\Desktop\其他重要\JGI\Protein_sequences"
os.makedirs(folder_path, exist_ok=True)
output_file = os.path.join(folder_path, "all.fasta")

# 读取 gene id
with open(input_file, "r") as f:
    gene_ids = [line.strip() for line in f if line.strip()]

def fetch_fasta(gene_id):
    url = f"https://img.jgi.doe.gov/cgi-bin/m/main.cgi?section=GeneDetail&page=genePageMainFaa&gene_oid={gene_id}"
    html = requests.get(url).text

    soup = BeautifulSoup(html, "html.parser")
    pre = soup.find("pre")
    if not pre:
        return None

    text = pre.get_text()

    # 去掉颜色字体控制字符
    text = re.sub(r"<.*?>", "", text)
    text = text.strip()

    return text

all_sequences = []

for gene_id in gene_ids:
    fasta = fetch_fasta(gene_id)
    if fasta and fasta.startswith(">"):
        all_sequences.append(fasta)
        print(f"✔ 已保存 {gene_id}")
    else:
        print(f"✘ 失败 {gene_id}")

with open(output_file, "w") as f:
    f.write("\n\n".join(all_sequences))

print(f"\n🎉 完成！所有序列已保存到： {output_file}")

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import re

id_file = r"C:\Users\localadmin\Desktop\其他重要\JGI\id.txt"
output_file = r"C:\Users\localadmin\Desktop\其他重要\JGI\Protein_sequences\all.fasta"

# 要从哪个 Gene ID 开始继续
restart_from = "8100270751"

# 读取所有 ID
with open(id_file, "r") as f:
    gene_ids = [line.strip() for line in f if line.strip()]

# 找到起始索引
if restart_from in gene_ids:
    start_index = gene_ids.index(restart_from)
else:
    raise ValueError(f"❗ 在 id.txt 中找不到 {restart_from}")

print(f"➡ 从第 {start_index+1} 个 ID 开始（ID = {restart_from}）")

# 只保留需要继续部分
gene_ids = gene_ids[start_index:]

def fetch_fasta(gene_id):
    url = f"https://img.jgi.doe.gov/cgi-bin/m/main.cgi?section=GeneDetail&page=genePageMainFaa&gene_oid={gene_id}"
    html = requests.get(url).text

    soup = BeautifulSoup(html, "html.parser")
    pre = soup.find("pre")
    if not pre:
        return None

    text = pre.get_text()
    text = re.sub(r"<.*?>", "", text)  # 移除HTML标签
    return text.strip()

# 🚨 使用 **追加模式** 写文件，前面的内容不会丢
with open(output_file, "a") as f:
    for gid in gene_ids:
        fasta = fetch_fasta(gid)
        if fasta and fasta.startswith(">"):
            f.write("\n" + fasta + "\n\n")
            print(f"✔ 已追加 {gid}")
        else:
            print(f"✘ 失败 {gid}")

print("\n🎉 继续下载完成！已追加到 all.fasta 不会覆盖之前内容。")


## 交集保留
   以“文件2”的序列名称为检索键，在“文件1+”中筛选，仅保留名称出现在“文件2”中的序列，形成目标集合。

In [36]:
from Bio import SeqIO
import os

# ===== 路径设置 =====
file1_plus = r"C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\merged_all_plus.fasta"  # 文件1+
file2_summary = r"C:\Users\localadmin\Desktop\其他重要\summary.txt"  # 文件2
output_fasta = r"C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\filtered_by_summary.fasta"

# ===== 步骤1：从 summary.txt 里提取目标 ID =====
target_ids = set()

with open(file2_summary, "r", encoding="utf-8") as summary:
    for line in summary:
        line = line.strip()
        # 只处理包含 '>' 的行（类似 FASTA header）
        if ">" in line:
            # 取 '>' 后面的内容
            after = line.split(">", 1)[1].strip()
            # ID 是 header 的第一段（遇到空格/制表符就截断）
            seq_id = after.split()[0]
            if seq_id:
                target_ids.add(seq_id)

print(f"[INFO] 从文件2中提取到 {len(target_ids)} 个目标序列ID。示例：")
for i, tid in enumerate(list(target_ids)[:5], 1):
    print(f"  {i}. {tid}")
if len(target_ids) == 0:
    print("警告：summary.txt 里没有解析到任何 ID，检查格式是否和预期一致。")

# ===== 步骤2：在文件1+中过滤这些ID对应的序列 =====
selected_records = []

with open(file1_plus, "r", encoding="utf-8") as infile:
    for record in SeqIO.parse(infile, "fasta"):
        if record.id in target_ids:
            selected_records.append(record)

print(f"[INFO] 在文件1+中匹配到 {len(selected_records)} 条序列。")

# ===== 步骤3：写出筛选结果 =====
with open(output_fasta, "w", encoding="utf-8") as outfile:
    SeqIO.write(selected_records, outfile, "fasta")

print("✅ 交集筛选完成")
print(f"输入文件1+：{file1_plus}")
print(f"输入文件2：{file2_summary}")
print(f"输出文件：{output_fasta}")
print(f"最终保留：{len(selected_records)} 条序列")


[INFO] 从文件2中提取到 44596 个目标序列ID。示例：
  1. WP_311999423.1
  2. UniRef100_A0A397TQX6
  3. tr|A0A9P6JQ06|A0A9P6JQ06_9AGAR
  4. tr|A0A0N7H2P6|A0A0N7H2P6_PSEFL
  5. tr|A0AAP9CLE8|A0AAP9CLE8_PSEFL
[INFO] 在文件1+中匹配到 35832 条序列。
✅ 交集筛选完成
输入文件1+：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\merged_all_plus.fasta
输入文件2：C:\Users\localadmin\Desktop\其他重要\summary.txt
输出文件：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\filtered_by_summary.fasta
最终保留：35832 条序列


## 全局去重与一致性校验 再次以氨基酸序列为唯一判据进行去重与一致性检查，确保最终结果中不存在任何重复的氨基酸序列。

In [34]:
from collections import OrderedDict
from Bio import SeqIO
import os

# 输入文件：上一步筛选得到的结果
input_file = r"C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\filtered_by_summary.fasta"

# 输出文件路径
output_file = os.path.join(os.path.dirname(input_file), "final_unique_sequences.fasta")

# 使用 OrderedDict 保留首个出现的序列（保证顺序）
unique_records = OrderedDict()

# 去重逻辑：以氨基酸序列为唯一判据
for record in SeqIO.parse(input_file, "fasta"):
    seq_str = str(record.seq).strip()
    if seq_str not in unique_records:
        unique_records[seq_str] = record

# 写出结果
SeqIO.write(unique_records.values(), output_file, "fasta")

# 统计信息输出
total = len(list(SeqIO.parse(input_file, "fasta")))
unique = len(unique_records)
print("✅ 全局去重与一致性校验完成")
print(f"输入文件：{input_file}")
print(f"输出文件：{output_file}")
print(f"原始序列数：{total}")
print(f"去重后序列数：{unique}")
print(f"重复移除数：{total - unique}")


✅ 全局去重与一致性校验完成
输入文件：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\filtered_by_summary.fasta
输出文件：C:\Users\localadmin\Desktop\其他重要\EBI\Protein_sequences\final_unique_sequences.fasta
原始序列数：35832
去重后序列数：35832
重复移除数：0


## 第一个数据库的其他代码

In [31]:
import os
import pandas as pd

# 文件夹路径
folder_path = r"C:\Users\localadmin\Desktop\其他重要\split result-1"

# 输出文件路径
output_file = os.path.join(folder_path, "merged_clean.xlsx")

# 结果列表
merged_list = []

for file in os.listdir(folder_path):
    if file.lower().endswith(".xlsx"):
        file_path = os.path.join(folder_path, file)
        try:
            # 先读整个文件
            raw = pd.read_excel(file_path, header=None, dtype=str)
            # 找到数据起始行（第一列中包含 "Query" 的行）
            start_idx = raw[raw.iloc[:,0].astype(str).str.contains("Query", na=False)].index
            if len(start_idx) == 0:
                print(f"⚠️ {file} 未找到数据起始行，跳过。")
                continue
            start_row = start_idx[0]

            # 重新读取有效数据部分
            df = pd.read_excel(file_path, header=start_row, dtype=str)
            merged_list.append(df)
            print(f"✅ 已合并：{file}（{len(df)} 行）")

        except Exception as e:
            print(f"❌ 解析失败 {file}：{e}")

# 合并所有文件
if merged_list:
    merged_df = pd.concat(merged_list, ignore_index=True)
    merged_df.to_excel(output_file, index=False)
    print(f"\n✅ 全部合并完成！")
    print(f"输出文件：{output_file}")
    print(f"总行数：{len(merged_df)}")
else:
    print("未找到可用数据。")


✅ 已合并：spli10.xlsx（915 行）
✅ 已合并：spli11.xlsx（1524 行）
✅ 已合并：spli12.xlsx（1468 行）
✅ 已合并：spli13.xlsx（1414 行）
✅ 已合并：spli14.xlsx（1393 行）
✅ 已合并：spli15.xlsx（882 行）
✅ 已合并：spli16.xlsx（956 行）
✅ 已合并：spli17.xlsx（1036 行）
✅ 已合并：spli18.xlsx（1013 行）
✅ 已合并：spli19.xlsx（960 行）
✅ 已合并：spli20.xlsx（976 行）
✅ 已合并：spli21.xlsx（957 行）
✅ 已合并：spli22.xlsx（964 行）
✅ 已合并：spli23.xlsx（1823 行）
✅ 已合并：spli24.xlsx（970 行）
✅ 已合并：spli25.xlsx（966 行）
✅ 已合并：spli26.xlsx（957 行）
✅ 已合并：spli27.xlsx（962 行）
✅ 已合并：spli28.xlsx（943 行）
✅ 已合并：spli29.xlsx（945 行）
✅ 已合并：spli30.xlsx（947 行）
✅ 已合并：spli31.xlsx（948 行）
✅ 已合并：spli32.xlsx（1113 行）
✅ 已合并：spli33.xlsx（950 行）
✅ 已合并：spli34.xlsx（914 行）
✅ 已合并：spli35.xlsx（969 行）
✅ 已合并：spli36.xlsx（948 行）
✅ 已合并：spli37.xlsx（940 行）
✅ 已合并：spli38.xlsx（930 行）
✅ 已合并：spli39.xlsx（1282 行）
✅ 已合并：spli41.xlsx（1760 行）
✅ 已合并：spli42.xlsx（1800 行）
✅ 已合并：spli43.xlsx（1802 行）
✅ 已合并：spli44.xlsx（1776 行）
✅ 已合并：spli45.xlsx（1803 行）
✅ 已合并：spli46.xlsx（1767 行）
✅ 已合并：spli47.xlsx（1240 行）
✅ 已合并：spli48.xlsx（1422 行）
✅ 已合并：spli49.xlsx（1245 行）
✅ 已合并：s

In [32]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import re

# ===== 配置 =====
input_file = r"C:\Users\localadmin\Desktop\其他重要\split result-1\merged_clean.xlsx"
output_folder = os.path.dirname(input_file)
filtered_csv = os.path.join(output_folder, "merged_filtered.csv")
filtered_txt = os.path.join(output_folder, "merged_results.txt")

# 筛选关键词（不区分大小写）
TARGETS_REGEX = r"(?:GMC_oxred_C|GMC_oxred_N|BBE|FAD)"

# ===== 读取 Excel =====
df = pd.read_excel(input_file, dtype=str).fillna("")

# 确保列名干净
df.columns = [c.strip() for c in df.columns]

# 检查关键列
required_cols = {"Short name", "Incomplete"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"❌ 文件缺少必要列：{required_cols - set(df.columns)}")

print(f"读取成功：{len(df)} 行")

# ===== 筛选逻辑 =====
filtered = df[
    (~df["Incomplete"].str.strip().isin(["N", "C"])) &  # 去掉 Incomplete=N 或 C
    (df["Short name"].str.contains(TARGETS_REGEX, flags=re.IGNORECASE, na=False))  # Short name 匹配关键词
]

print(f"筛选后剩余 {len(filtered)} 行")

# ===== 输出结果 =====
filtered.to_csv(filtered_csv, index=False)
filtered.to_csv(filtered_txt, sep="\t", index=False)

print("✅ 输出完成")
print(f"CSV 文件：{filtered_csv}")
print(f"TXT 文件：{filtered_txt}")


读取成功：271786 行
筛选后剩余 46247 行
✅ 输出完成
CSV 文件：C:\Users\localadmin\Desktop\其他重要\split result-1\merged_filtered.csv
TXT 文件：C:\Users\localadmin\Desktop\其他重要\split result-1\merged_results.txt
